# PyMuPDF

In [1]:
import pymupdf

In [2]:
filename = "input_dict_files/rodegem_rundi1970_o.pdf"

rundi_dict_pdf = pymupdf.open(filename)  # or pymupdf.Document(filename)

In [3]:
rundi_sample_page = rundi_dict_pdf.load_page(27)

In [4]:
rundi_sample_page.get_text().split('\n')

['A',
 'a, (inv.), interjection de négation, de dénéga\xad',
 'tion : non. Peut être prononcée à bouche entrou\xad',
 'verte ou à bouche fermée. Dans ce dernier cas, ',
 'elle est souvent redoublée en a a, le second ',
 'étant prononcé légèrement plus haut.',
 'umwaba, imy-, 3|4, surface de terre arable dans ',
 'le marais. Mw-irima ry’imyaba, durant la cul\xad',
 'ture des marais.',
 'akâba, utw-, 12|13, lopin de terre arable. Drain ',
 "dans les champs de marais. Akâba n'umugazo ",
 'barekérwa mu busïze bw’îmfûnzo, dans le ter\xad',
 'rain défriché, parmi les papyrus on dégage un ',
 'drain appelé akâba.',
 'kwâba, -âvye, crier vers; avoir recours à, ',
 'recourir à. Hàgira ikibâye gishikiye umuryango ',
 'bàca bâgenda kwâba umupfùmu, si un malheur ',
 'frappait la famille, ils allaient aussitôt consulter ',
 'le devin.',
 'kwabàba, -vye, (-ab--), tendre la main pour ',
 'prendre en se dressant sur la pointe des pieds. ',
 'Il Commencer à ramper (enfant). Syn. kwavüra. ',
 '|( ',
 'E

In [12]:
import re

def insert_start_end_tokens(text: str) -> str:
    """
    Inserts '<start>' before each dictionary entry.

    Assumes each entry starts at a new line and begins with 2
    comma-separated fields, with the second .
    """

    # Step 1: insert <start>
    start_pattern = re.compile(
        r'^((?:[^,\n]+,\s*){1,2}\d+(?:\|\d+)?)',
        re.MULTILINE
    )
    text = start_pattern.sub(r'<start>\1', text)

    # # Step 2: for each <start>, find first '.' and insert <end>
    # def add_end_token(match):
    #     segment = match.group(0)
    #     dot_index = segment.find('.')
    #     if dot_index != -1:
    #         return segment[:dot_index + 1] + '<end>' + segment[dot_index + 1:]
    #     return segment  # no full stop found

    # Match from <start> up to next <start> or end of text
    entry_pattern = re.compile(r'<start>.*?(?=<start>|$)', re.DOTALL)
    # text = entry_pattern.sub(add_end_token, text)

    return text

In [16]:
# result = insert_start_end_tokens(rundi_sample_page.get_text())
# insert_start_end_tokens(rundi_sample_page.get_text())

"A\na, (inv.), interjection de négation, de dénéga\xad\ntion : non. Peut être prononcée à bouche entrou\xad\nverte ou à bouche fermée. Dans ce dernier cas, \nelle est souvent redoublée en a a, le second \nétant prononcé légèrement plus haut.\n<start>umwaba, imy-, 3|4, surface de terre arable dans \nle marais. Mw-irima ry’imyaba, durant la cul\xad\nture des marais.\n<start>akâba, utw-, 12|13, lopin de terre arable. Drain \ndans les champs de marais. Akâba n'umugazo \nbarekérwa mu busïze bw’îmfûnzo, dans le ter\xad\nrain défriché, parmi les papyrus on dégage un \ndrain appelé akâba.\nkwâba, -âvye, crier vers; avoir recours à, \nrecourir à. Hàgira ikibâye gishikiye umuryango \nbàca bâgenda kwâba umupfùmu, si un malheur \nfrappait la famille, ils allaient aussitôt consulter \nle devin.\nkwabàba, -vye, (-ab--), tendre la main pour \nprendre en se dressant sur la pointe des pieds. \nIl Commencer à ramper (enfant). Syn. kwavüra. \n|( \nEtre presque \négal, approcher \nde prés. \nUwàbâba umukù

In [17]:
# for entry in result.split('<start>'):
#     print(entry)
#     print('\n')

A
a, (inv.), interjection de négation, de dénéga­
tion : non. Peut être prononcée à bouche entrou­
verte ou à bouche fermée. Dans ce dernier cas, 
elle est souvent redoublée en a a, le second 
étant prononcé légèrement plus haut.



umwaba, imy-, 3|4, surface de terre arable dans 
le marais. Mw-irima ry’imyaba, durant la cul­
ture des marais.



akâba, utw-, 12|13, lopin de terre arable. Drain 
dans les champs de marais. Akâba n'umugazo 
barekérwa mu busïze bw’îmfûnzo, dans le ter­
rain défriché, parmi les papyrus on dégage un 
drain appelé akâba.
kwâba, -âvye, crier vers; avoir recours à, 
recourir à. Hàgira ikibâye gishikiye umuryango 
bàca bâgenda kwâba umupfùmu, si un malheur 
frappait la famille, ils allaient aussitôt consulter 
le devin.
kwabàba, -vye, (-ab--), tendre la main pour 
prendre en se dressant sur la pointe des pieds. 
Il Commencer à ramper (enfant). Syn. kwavüra. 
|( 
Etre presque 
égal, approcher 
de prés. 
Uwàbâba umukùru, vice-président, lieutenant. 
Aramwabâba mu 

# Tesseract

In [1]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageOps, ImageFilter
import time

In [2]:
def ocr_pdf_pipeline(pdf_path, image_dpi=300, languages ="fra+latn", ocr_config = '--oem 3 --psm 3' ):
    """
    Extracts text page-by-page to keep memory usage low.
    """
    full_text = []

    # convert_from_path returns a list of PIL images. 
    # To be truly memory efficient with large PDFs, we process them one by one.
    try:
        # We use a generator-like approach or process in small chunks
        # thread_count can speed up conversion, but increases CPU/RAM usage
        images = convert_from_path(pdf_path, dpi=image_dpi) 

        for i, image in enumerate(images):
            print(f"Processing page {i + 1}...")
            
            # 1. Perform OCR on the single page image
            
            # Specify languages: e.g., 'fra+yor' or 'fra+latn'
            # 'fra' handles the French vocabulary/grammar
            # 'latn' handles Latin characters/accents not specific to French

            # --- CROP LOGIC ---
            width, height = image.size
            
            crop_box = (
                width * 0.05,  # 5% from left
                height * 0.05, # 5% from top
                width * 0.95,  # 95% across (5% from right)
                height * 0.95  # 95% down (5% from bottom)
            )
            
            cropped_image = image.crop(crop_box)
           
            # if you wanna see the images
            cropped_image.save(f'sample_page_{i}.jpg')

            # Execute OCR
            text = pytesseract.image_to_string(cropped_image, lang=languages, config=ocr_config)
        
            # 2. Append text to our list
            full_text.append(text)
            
            # 3. Explicitly clear the image from memory
            cropped_image.close()
            image.close()     
            break
        return "\n\n".join(full_text)

    except Exception as e:
        return f"An error occurred: {e}"

In [33]:
# import os
# os.environ['TESSDATA_PREFIX'] = '/Users/harshamarsha/Documents/CourseWork/Bantu_HiWi/digitizing/tessdata_best/'

In [21]:
filename = "input_dict_files/rundi_sample.pdf"

start_time = time.perf_counter()
extracted_content = ocr_pdf_pipeline(
    
    filename, 
                                     
    image_dpi=900, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = '--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0'
)

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time):.2f} seconds')

Processing page 1...
Time taken to OCR: 18.21 seconds


In [22]:
print(extracted_content)

a, (inv.), interjection de négation, de dénéga-
tion: non. Peut être prononcée à bouche entrou-
verte ou à bouche fermée. Dans ce dernier cas,
eke est souvent redoublée en a a, le second
étant prononcé légèrement plus haut.

umwaba, imy-, 34, surface de terre arable dans
le marais, Mw-irima ryimyvaba, durant ta cul-
fure des maårais.

akāba, utw-, 12|13, lopin de terre arable. Drain
dans les champs de marais. Akaba n'umugazo
barekérwa mu busříze bw'îmfůnzo, dans le ter-
rain défriché, parmi les papyrus on dégage un
drain appelé akāba.

kwâba, -âvye, crier vers; avoir recours ğ,
recourir à. Hšgira ikibâye gishikíye umuryango
băca bågenda kwâôba umupfůmu, si un malheur
frappait la famille, ils allaient aussitôt consulier
le devin.

kwabāäba, -vye, (-ab--), tendre la main pour
prendre en se dressant sur la pointe des pieds.
li Commencer à ramper (enfant). Syn. kwavūra.
| Etre presque égal, approcher de près.
Uwăbāba umukůru, vice-président, lieutenant.
Aramwabāba mu kuvůka, is ont presque

# Tesseract + Line segmentation

In [1]:

import time
import os
import cv2
import numpy as np
import pytesseract
from pytesseract import Output
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageFilter
from tqdm.notebook import tqdm

import gc # Garbage Collector


# Set Tesseract to use multiple threads internally
os.environ["OMP_THREAD_LIMIT"] = "8"

In [17]:
def save_entry_to_file(entry_str, file_path="dictionary_output.txt"):
    """Appends a single entry to the text file immediately."""
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")
        

def find_footer_separator_cv2(pil_img):
    # 1. Convert to OpenCV grayscale
    img = np.array(pil_img.convert('L'))
    h, w = img.shape
    
    # 2. Focus on the bottom 30% and binarize
    # We use a higher threshold (220) to catch very faint/thin lines
    start_y = int(h * 0.50)
    roi = img[start_y:, :]
    _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Create a horizontal "needle" kernel
    # This kernel is 1 pixel tall but 50 pixels wide.
    # It acts like a comb that only lets horizontal structures through.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
    
    # 4. Apply Morphology (Erosion followed by Dilation)
    # This deletes everything that isn't at least 50 pixels wide horizontally.
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
    # 5. Find the Y-coordinate of the remaining pixels
    # We look for the row with the most white pixels in the 'detected_lines' mask
    row_sums = np.sum(detected_lines, axis=1)
    
    if np.max(row_sums) > 0:
        # Find the index of the row with the maximum "line-ness"
        line_relative_y = np.argmax(row_sums)
        global_y = start_y + line_relative_y
        
        # Buffer: return 15 pixels above the line to ensure it's fully gone
        return global_y - 15
        
    return h

    
def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn", ocr_config='--oem 3 --psm 3'):
    
    output_file = "ocr_results.txt"
    # Clear file if it exists from a previous run
    if os.path.exists(output_file):
        os.remove(output_file)
    
    # Get page count without loading images
    info = pdfinfo_from_path(pdf_path)
    max_pages = info["Pages"]
    
    for page_num in tqdm(range(1, max_pages + 1), desc="OCR Progress", unit="page"): 
        
        print(f"Processing page {page_num}/{max_pages}...")
        
        images = convert_from_path(pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num)
        if not images:
            continue
            
        image = images[0]

        # 1. Detect and Crop Footer
        footer_y = find_footer_separator_cv2(image)
        if footer_y < image.size[1]:
            image = image.crop((0, 0, image.size[0], footer_y))   
                
        width, height = image.size
        left_m, right_m = width * 0.05, width * 0.95
        top_m, mid = height * 0.05, width / 2
        
        columns = [
            ("Left", image.crop((left_m, top_m, mid, height))),
            ("Right", image.crop((mid, top_m, right_m, height)))
        ]

        for col_name, col_img in columns:
            # debug image
            col_img.save(f'p{page_num}_{col_name}.png')
            
            data = pytesseract.image_to_data(col_img, lang=languages, config=ocr_config, output_type=Output.DICT)
            
            # Step 1: Reconstruct lines
            lines = []
            n_words = len(data['text'])
            current_line_text, current_line_x = [], None

            for j in range(n_words):
                if int(data['conf'][j]) > 0:
                    if int(data['word_num'][j]) == 1:
                        if current_line_text:
                            lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})
                        current_line_x = data['left'][j]
                        current_line_text = [data['text'][j]]
                    else:
                        current_line_text.append(data['text'][j])
            
            if current_line_text:
                lines.append({'x': current_line_x, 'text': " ".join(current_line_text)})

            if not lines:
                continue

            # Step 2: Thresholding
            median_x = np.median([l['x'] for l in lines])
            indent_threshold = 40 # Adjust to 90 if 40 is too sensitive for 600dpi

            # Step 3: Group and Write to File
            current_entry = []
            for line in lines:
                is_indent = line['x'] > (median_x + indent_threshold)

                if is_indent:
                    if current_entry:
                        # --- SAVE POSITION 1: End of an entry detected by a new indent ---
                        entry_str = " ".join(current_entry).strip()
                        save_entry_to_file(entry_str, output_file)
                    
                    current_entry = [line['text']]
                else:
                    if current_entry:
                        current_entry.append(line['text'])
                    else:
                        current_entry = [line['text']]

            if current_entry:
                # --- SAVE POSITION 2: Final entry of the column ---
                entry_str = " ".join(current_entry).strip()
                save_entry_to_file(entry_str, output_file)

        # Cleanup Memory
        for _, c_img in columns:
            c_img.close()
        image.close()
        del images 
        if page_num == 4: break
    print(f"Finished! All entries saved to {output_file}")
    return True

In [25]:
# --- KEEP YOUR HELPER FUNCTIONS ---
def save_entry_to_file(entry_str, file_path="ocr_results.txt"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")
def find_footer_separator_cv2(pil_img):
    # 1. Convert to OpenCV grayscale
    img = np.array(pil_img.convert('L'))
    h, w = img.shape
    
    # 2. Focus on the bottom 70% and binarize
    # We use a higher threshold (220) to catch very faint/thin lines
    start_y = int(h * 0.30)
    roi = img[start_y:, :]
    _, binary = cv2.threshold(roi, 220, 255, cv2.THRESH_BINARY_INV)
    
    # 3. Create a horizontal "needle" kernel
    # This kernel is 1 pixel tall but 50 pixels wide.
    # It acts like a comb that only lets horizontal structures through.
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (100, 1))
    
    # 4. Apply Morphology (Erosion followed by Dilation)
    # This deletes everything that isn't at least 100 pixels wide horizontally.
    detected_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    
    # 5. Find the Y-coordinate of the remaining pixels
    # We look for the row with the most white pixels in the 'detected_lines' mask
    row_sums = np.sum(detected_lines, axis=1)
    
    if np.max(row_sums) > 0:
        # Find the index of the row with the maximum "line-ness"
        line_relative_y = np.argmax(row_sums)
        global_y = start_y + line_relative_y
        
        return global_y - 10
        
    return h

def find_dynamic_split_point(pil_img):
    # 1. Convert to grayscale and binarize
    bw_arr = np.array(pil_img.convert('L'))
    # 0 = black (text), 1 = white (background)
    bw_arr = (bw_arr > 128).astype(np.uint8) 
    
    h, w = bw_arr.shape
    
    # 2. Focus on the middle 20% of the page width
    # This ensures we don't accidentally split on a margin
    search_start = int(w * 0.35)
    search_end = int(w * 0.65)
    mid_zone = bw_arr[:, search_start:search_end]
    
    # 3. Vertical Projection: Sum the white pixels in each column
    # The gutter will be the column with the MOST white pixels
    column_white_sums = np.sum(mid_zone, axis=0)
    
    # Find the index of the column that is "whitest"
    best_gutter_relative = np.argmax(column_white_sums)
    
    # 4. Return the absolute X coordinate
    dynamic_mid = search_start + best_gutter_relative
    
    # Debug: if the "whitest" column isn't significantly white, fallback to w/2
    if column_white_sums[best_gutter_relative] < (h * 0.95):
        return int(w / 2)
        
    return dynamic_mid



def ocr_pdf_lineseg_pipeline(pdf_path, image_dpi=600, languages="fra+latn", ocr_config='--oem 3 --psm 3', output_file = "ocr_results.txt", export_pics=False, limit_max_p = None):
    
    # Check for total pages
    info = pdfinfo_from_path(pdf_path)
    max_pages = info["Pages"]
    
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("")        
        
    for page_num in tqdm(range(1, max_pages + 1), desc="Processing Dictionary", unit="pg"):
        # if page_num not in [12,14,15,16]:continue # <<----------- Testing
            
        try:
            # --- SINGLE PAGE RUN: Load only 1 page at a time ---
            images = convert_from_path(pdf_path, 
                                       dpi=image_dpi, 
                                       first_page=page_num, 
                                       last_page=page_num)
            if not images: continue
            img = images[0]

            # 1. Detect Footer (OpenCV Horizontal Kernel)
            footer_y = find_footer_separator_cv2(img)
            if footer_y < img.size[1]:
                img = img.crop((0, 0, img.size[0], footer_y))

            # 2. Split Columns
            w, h = img.size
            dynamic_mid = find_dynamic_split_point(img)
            columns = [
                # Left: Margin to the dynamic gutter
                img.crop((w * 0.05, h * 0.05, dynamic_mid * 1.02, h)),
                # Right: Dynamic gutter to the right margin
                img.crop((dynamic_mid, h * 0.05, w * 0.95, h))
            ]
            
            for i, col_img in enumerate(columns):
                if export_pics==True:
                    col_img.save(f'p{page_num}_{i}.png')
                # Noise cleanup
                # col_img = col_img.filter(ImageFilter.RankFilter(size=3, rank=7))
                
                # OCR Data
                data = pytesseract.image_to_data(col_img, lang=languages, 
                                                 config=ocr_config, output_type=Output.DICT)
                
                # --- LINE RECONSTRUCTION ---
                lines = []

                curr_text, curr_x = [], None
            
                for j in range(len(data['text'])):
                    if int(data['conf'][j]) > 0:
                        if int(data['word_num'][j]) == 1:
                            if curr_text:
                                lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                            curr_x, curr_text = data['left'][j], [data['text'][j]]
                        else:
                            curr_text.append(data['text'][j])
                if curr_text: lines.append({'x': curr_x, 'text': " ".join(curr_text)})
                if not lines: continue

                # write to file in append mode
                with open(output_file, "a", encoding="utf-8") as f:
                # --- ENTRY GROUPING & WRITING ---
                    median_x = np.median([l['x'] for l in lines])
                    current_entry = []
                    for line in lines:
                        if line['x'] > (median_x + 40): # Indent check
                            if current_entry:
                                f.write(" ".join(current_entry).strip() + "\n\n")
                            current_entry = [line['text']]
                        else:
                            current_entry.append(line['text'])
                    if current_entry:
                        f.write(" ".join(current_entry).strip() + "\n\n")
                        
                col_img.close()

            # cleanup
            img.close()
            del images
            del img
            gc.collect()
            
        except Exception as e:
            print(f"Skipping page {page_num} due to error: {e}")
            continue
        if limit_max_p and page_num == limit_max_p :break # <<----------- Testing
                
    return f"Success! Results appended to {output_file}"

In [27]:
# filename = "input_dict_files/rundi_sample.pdf" # sample with 3 pages
filename = "input_dict_files/rundi_working_file.pdf" # main

start_time = time.perf_counter()
extracted_content = ocr_pdf_lineseg_pipeline(
    
    filename, 
                                     
    image_dpi=600, 
                                     
    languages = "script/Latin", 
                                     
    ocr_config = '--oem 3 --psm 3 -c load_system_dawg=0 -c load_freq_dawg=0',

    output_file = "rundi_extracted.txt",

    export_pics = True, #debug

    limit_max_p = 4
)

end_time = time.perf_counter()

print(f'Time taken to OCR: {(end_time-start_time)/60:.2f} min')
extracted_content

Processing Dictionary:   0%|          | 0/588 [00:00<?, ?pg/s]

Time taken to OCR: 0.86 min


'Success! Results appended to rundi_extracted.txt'